# Import Library

In [1]:
import pandas as pd
from pathlib import Path

# Load Data

Baca file raw

In [23]:
file_path = "./data/raw/raw_xaw_idr_june_23_26.csv"
# text = Path(file_path).read_text(encoding="utf-8-sig")

Bersihkan quote luar tiap baris

In [25]:
# clean_lines = []

# for line in text.splitlines():
#     line = line.strip()
    
#     if line.startswith('"') and line.endswith('"'):
#         line = line[1:-1]
    
#     line = line.replace('""', '"')
#     clean_lines.append(line)

# clean_text = "\n".join(clean_lines)


Load clean data

In [26]:
# 3. Load ulang sebagai CSV normal
df = pd.read_csv(file_path)

df.head()

,Date,Price,Open,High,Low,Vol.,Change %
0,06/29/2026,"717,131","731,647","728,815","716,386",NaN,-1.44%
1,06/28/2026,"727,635","731,647","732,456","726,602",NaN,-0.64%
2,06/26/2026,"732,303","721,550","733,941","716,132",NaN,1.48%
3,06/25/2026,"721,599","717,004","724,790","711,128",NaN,0.64%
4,06/24/2026,"717,017","733,506","735,639","709,917",NaN,-2.24%


In [27]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 784 entries, 0 to 783
Data columns (total 7 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   Date      784 non-null    object 
 1   Price     784 non-null    object 
 2   Open      784 non-null    object 
 3   High      784 non-null    object 
 4   Low       784 non-null    object 
 5   Vol.      0 non-null      float64
 6   Change %  784 non-null    object 
dtypes: float64(1), object(6)
memory usage: 43.0+ KB


# Data preprocessing

## Rename column

In [28]:
df = df.rename(columns=
{
    "Date": "timestamp",
    "Price": "close",
    "Open": "open",
    "High": "high",
    "Low": "low",
    "Vol.": "volume"
})

In [29]:
print(df.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 784 entries, 0 to 783
Data columns (total 7 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   timestamp  784 non-null    object 
 1   close      784 non-null    object 
 2   open       784 non-null    object 
 3   high       784 non-null    object 
 4   low        784 non-null    object 
 5   volume     0 non-null      float64
 6   Change %   784 non-null    object 
dtypes: float64(1), object(6)
memory usage: 43.0+ KB
None


## Bersihkan angka ribuan

In [30]:
price_cols = ["open", "high", "low", "close"]
for col in price_cols:
    df[col] = (
        df[col]
        .astype(str)
        .str.replace(",", "", regex=False)
        .astype(float)
    )

## Convert timestamp

In [31]:
df["timestamp"] = pd.to_datetime(df["timestamp"])

In [32]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 784 entries, 0 to 783
Data columns (total 7 columns):
 #   Column     Non-Null Count  Dtype         
---  ------     --------------  -----         
 0   timestamp  784 non-null    datetime64[ns]
 1   close      784 non-null    float64       
 2   open       784 non-null    float64       
 3   high       784 non-null    float64       
 4   low        784 non-null    float64       
 5   volume     0 non-null      float64       
 6   Change %   784 non-null    object        
dtypes: datetime64[ns](1), float64(5), object(1)
memory usage: 43.0+ KB


## Isi dengan dummy data

In [33]:
# Volume tidak tersedia, isi dummy
df["volume"] = 0.0
df["amount"] = 0.0

In [34]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 784 entries, 0 to 783
Data columns (total 8 columns):
 #   Column     Non-Null Count  Dtype         
---  ------     --------------  -----         
 0   timestamp  784 non-null    datetime64[ns]
 1   close      784 non-null    float64       
 2   open       784 non-null    float64       
 3   high       784 non-null    float64       
 4   low        784 non-null    float64       
 5   volume     784 non-null    float64       
 6   Change %   784 non-null    object        
 7   amount     784 non-null    float64       
dtypes: datetime64[ns](1), float64(6), object(1)
memory usage: 49.1+ KB


## Sort lama ke baru

In [35]:
df = df.sort_values("timestamp").reset_index(drop=True)

In [36]:
df.head()

,timestamp,close,open,high,low,volume,Change %,amount
0,2023-06-29,285963.0,285954.0,286829.0,283809.0,0.0,0.04%,0.0
1,2023-06-30,287747.0,286146.0,288290.0,284941.0,0.0,0.62%,0.0
2,2023-07-03,288583.0,287765.0,290147.0,286993.0,0.0,0.29%,0.0
3,2023-07-04,288571.0,288733.0,289488.0,288428.0,0.0,0.00%,0.0
4,2023-07-05,287413.0,288741.0,290565.0,287506.0,0.0,-0.40%,0.0


## Ambil kolom untuk Kronos

In [37]:
kronos_df = df[["timestamp", "open", "high", "low", "close", "volume", "amount"]].copy()

In [38]:
kronos_df.head()

,timestamp,open,high,low,close,volume,amount
0,2023-06-29,285954.0,286829.0,283809.0,285963.0,0.0,0.0
1,2023-06-30,286146.0,288290.0,284941.0,287747.0,0.0,0.0
2,2023-07-03,287765.0,290147.0,286993.0,288583.0,0.0,0.0
3,2023-07-04,288733.0,289488.0,288428.0,288571.0,0.0,0.0
4,2023-07-05,288741.0,290565.0,287506.0,287413.0,0.0,0.0


## Hapus missing value kalau ada

In [39]:
kronos_df = kronos_df.dropna()
kronos_df.head()

,timestamp,open,high,low,close,volume,amount
0,2023-06-29,285954.0,286829.0,283809.0,285963.0,0.0,0.0
1,2023-06-30,286146.0,288290.0,284941.0,287747.0,0.0,0.0
2,2023-07-03,287765.0,290147.0,286993.0,288583.0,0.0,0.0
3,2023-07-04,288733.0,289488.0,288428.0,288571.0,0.0,0.0
4,2023-07-05,288741.0,290565.0,287506.0,287413.0,0.0,0.0


In [40]:
kronos_df.to_csv("./data/preprocessing/prep_june_23_26.csv", index=False)